[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C44_Adversarial_Security_Course/01_adversarial_examples/01_adversarial_examples.ipynb)

# 01 · 对抗样本：FGSM、PGD 与对抗训练（用 numpy）

目标：在玩具 **2 层 MLP** 分类器（纯 numpy）上从零实现 **FGSM**、**PGD** 对抗攻击，用 **对抗训练** 防御，并演示 **梯度遮蔽** 这一评测陷阱。

> **防御视角**：我们造对抗样本，是为了度量脆弱性、验证防御。所有演示在合成数据（two-moons）+ 玩具模型上，规模极小。

> 为什么用 MLP 而非线性模型？线性分类器的边界只是一个超平面，**对抗训练能改善的空间有限**；一个小 MLP（非线性边界）才能清楚、可复现地展示对抗训练把鲁棒精度拉回来——这也更贴近 Madry 等人在神经网络上的原始设定。

路线：MLP 靶子 + 输入梯度(反向传播) → FGSM → PGD（投影+重启）→ 扰动预算曲线 → 对抗训练 → ✏️ 练习(定向FGSM/PGD一步/弱攻击陷阱/对抗训练) → 📖 答案 → 🧪 梯度遮蔽胶囊。

## 1 · 靶子（2 层 MLP）与「损失对输入的梯度」

对抗攻击的核心原料是 **损失对输入 x 的梯度** $\nabla_x L$（不是对参数的梯度！）。

用 two-moons（非线性可分）合成数据训一个 `tanh` 隐层的 2 层 MLP。攻击需要把梯度一路**反向传播到输入**：$\frac{\partial L}{\partial x} = \big((p-y)W_2^\top \odot (1-\tanh^2 z_1)\big)W_1^\top$。先用数值梯度校验它正确。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_moons(n=400, noise=0.18, seed=0):
    r = np.random.default_rng(seed); n0=n//2; n1=n-n0
    t0 = np.pi*r.uniform(0,1,n0); X0 = np.stack([np.cos(t0), np.sin(t0)],1)
    t1 = np.pi*r.uniform(0,1,n1); X1 = np.stack([1-np.cos(t1), 1-np.sin(t1)-0.5],1)
    X = np.vstack([X0,X1]) + noise*r.standard_normal((n,2))
    y = np.concatenate([np.zeros(n0), np.ones(n1)]).astype(int)
    p = r.permutation(n); return X[p], y[p]

def sigmoid(z): return 1.0/(1.0+np.exp(-z))

class MLP:
    '''2 层 MLP：tanh 隐层 + sigmoid 输出。可算参数梯度(训练)与输入梯度(攻击)。'''
    def __init__(s, d=2, h=16, seed=0):
        r = np.random.default_rng(seed)
        s.W1 = r.standard_normal((d,h))*0.5; s.b1 = np.zeros(h)
        s.W2 = r.standard_normal((h,1))*0.5; s.b2 = np.zeros(1)
    def forward(s, X):
        s.z1 = X@s.W1 + s.b1; s.a1 = np.tanh(s.z1)
        s.z2 = s.a1@s.W2 + s.b2; return sigmoid(s.z2).ravel()
    def prob(s, X): return s.forward(X)
    def predict(s, X): return (s.prob(X) > 0.5).astype(int)
    def grad_input(s, X, y):
        '''dL/dx，逐样本。返回 (n,d)。'''
        p = s.forward(X)
        dz2 = (p - y)[:, None]                 # (n,1)
        dz1 = (dz2 @ s.W2.T) * (1 - np.tanh(s.z1)**2)   # (n,h)
        return dz1 @ s.W1.T                    # (n,d)
    def fit(s, X, y, lr=0.15, epochs=700, l2=1e-4):
        n = len(y)
        for _ in range(epochs):
            p = s.forward(X)
            dz2 = (p - y)[:, None] / n
            dW2 = s.a1.T@dz2 + l2*s.W2; db2 = dz2.sum(0)
            dz1 = (dz2 @ s.W2.T) * (1 - np.tanh(s.z1)**2)
            dW1 = X.T@dz1 + l2*s.W1; db1 = dz1.sum(0)
            s.W2 -= lr*dW2; s.b2 -= lr*db2; s.W1 -= lr*dW1; s.b1 -= lr*db1
        return s

def acc(m, X, y): return float(np.mean(m.predict(X) == y))

X, y = make_moons()
Xtr, ytr, Xte, yte = X[:300], y[:300], X[300:], y[300:]
clf = MLP().fit(Xtr, ytr)
clean = acc(clf, Xte, yte)
print(f'干净测试精度 = {clean:.3f}')
assert clean > 0.82, 'MLP 在 moons 上应学得不错'
# 数值梯度校验：解析输入梯度 vs 有限差分
i = 0; h = 1e-5
g = clf.grad_input(Xte[i:i+1], yte[i:i+1])[0]
def loss1(x):
    pp = np.clip(clf.prob(x[None])[0], 1e-9, 1-1e-9); yy = yte[i]
    return -(yy*np.log(pp) + (1-yy)*np.log(1-pp))
gnum = np.array([(loss1(Xte[i]+h*np.eye(2)[k]) - loss1(Xte[i]-h*np.eye(2)[k]))/(2*h) for k in range(2)])
assert np.allclose(g, gnum, atol=1e-3), '解析输入梯度应与数值梯度一致'
print('✅ 损失对输入的梯度校验通过（反向传播到输入 == 数值梯度）')

## 2 · FGSM：单步符号梯度攻击

$x_{adv} = x + \epsilon\cdot\mathrm{sign}(\nabla_x L)$。沿「让损失上升最快」方向，在 L∞ 预算 ε 内走满一步。

验证：FGSM 后测试精度应**显著下降**（攻击奏效）。

In [ ]:
def fgsm(model, X, y, eps):
    g = model.grad_input(X, y)
    return X + eps * np.sign(g)            # L∞: 每维顶到 ±eps

eps = 0.5
Xadv = fgsm(clf, Xte, yte, eps)
fgsm_acc = acc(clf, Xadv, yte)
print(f'干净精度 {clean:.3f}  →  FGSM(ε={eps}) 后 {fgsm_acc:.3f}')
assert fgsm_acc < clean - 0.1, 'FGSM 应显著拉低精度'
print('✅ FGSM 攻击奏效：微小符号扰动即让精度大幅下降')

## 3 · PGD：多步迭代 + 投影 + 随机重启

把一大步拆成多小步，每步走 α 再 **投影**回 ε-球（L∞ 下即 clip 到 [x-ε, x+ε]），并用随机起点重启取最坏。

验证：在同样 ε 下，PGD 应当**不弱于 FGSM**（鲁棒精度 ≤ FGSM）。

In [ ]:
def pgd(model, X, y, eps, alpha=None, steps=20, restarts=3, seed=0):
    if alpha is None: alpha = eps / 4
    r = np.random.default_rng(seed)
    best = X.copy(); best_loss = -np.ones(len(y)) * np.inf
    def sample_loss(Xa):
        pp = np.clip(model.prob(Xa), 1e-9, 1-1e-9)
        return -(y*np.log(pp) + (1-y)*np.log(1-pp))
    for _ in range(restarts):
        Xa = X + r.uniform(-eps, eps, X.shape)    # 随机起点
        Xa = np.clip(Xa, X - eps, X + eps)
        for _ in range(steps):
            g = model.grad_input(Xa, y)
            Xa = Xa + alpha * np.sign(g)
            Xa = np.clip(Xa, X - eps, X + eps)    # 投影回 ε-球
        l = sample_loss(Xa); upd = l > best_loss
        best[upd] = Xa[upd]; best_loss[upd] = l[upd]
    return best

Xpgd = pgd(clf, Xte, yte, eps=0.5)
pgd_acc = acc(clf, Xpgd, yte)
print(f'FGSM 后 {fgsm_acc:.3f}  |  PGD 后 {pgd_acc:.3f}  (ε=0.5)')
assert pgd_acc <= fgsm_acc + 1e-9, 'PGD 应不弱于 FGSM'
print('✅ PGD 不弱于 FGSM：迭代+重启找到更坏的扰动（评测应以强攻击为准）')

## 4 · 扰动预算曲线：鲁棒精度随 ε 单调下降

鲁棒性数字**离开 ε 就没有意义**。扫一组 ε，看 PGD 下精度如何随预算下降——这条曲线才是鲁棒性的完整刻画。

In [ ]:
eps_grid = [0.0, 0.1, 0.2, 0.3, 0.5, 0.8]
robust = []
for e in eps_grid:
    a = clean if e == 0 else acc(clf, pgd(clf, Xte, yte, eps=e), yte)
    robust.append(a); print(f'  ε={e:<4} 鲁棒精度={a:.3f}')
assert all(robust[i] >= robust[i+1] - 0.05 for i in range(len(robust)-1)), '鲁棒精度应随 ε 大体单调下降'
print('✅ 鲁棒精度随 ε 单调下降 —— 报告鲁棒性必须报告对应的 ε')

## 5 · 对抗训练：min-max 防御

$\min_\theta \mathbb{E}[\max_{\|\delta\|\le\epsilon} L(f_\theta(x+\delta), y)]$：每个 epoch 先用 PGD 生成最坏扰动，再在对抗样本上做一步参数更新。

验证：对抗训练后的模型在 PGD 下的鲁棒精度应**显著高于**普通模型（代价：干净精度通常略降）。

In [ ]:
def adversarial_train(X, y, eps, lr=0.15, epochs=700, l2=1e-4, inner_steps=10, seed=0):
    m = MLP(seed=seed); n = len(y)
    for _ in range(epochs):
        Xa = pgd(m, X, y, eps=eps, alpha=eps/4, steps=inner_steps, restarts=1)  # 内层攻击
        p = m.forward(Xa)                                                       # 外层在最坏点上更新
        dz2 = (p - y)[:, None] / n
        dW2 = m.a1.T@dz2 + l2*m.W2; db2 = dz2.sum(0)
        dz1 = (dz2 @ m.W2.T) * (1 - np.tanh(m.z1)**2)
        dW1 = Xa.T@dz1 + l2*m.W1; db1 = dz1.sum(0)
        m.W2 -= lr*dW2; m.b2 -= lr*db2; m.W1 -= lr*dW1; m.b1 -= lr*db1
    return m

rob = adversarial_train(Xtr, ytr, eps=0.5)
plain_robacc = acc(clf, pgd(clf, Xte, yte, eps=0.5), yte)
adv_robacc   = acc(rob, pgd(rob, Xte, yte, eps=0.5), yte)
print(f'普通模型: 干净={clean:.3f}  PGD鲁棒={plain_robacc:.3f}')
print(f'对抗训练: 干净={acc(rob,Xte,yte):.3f}  PGD鲁棒={adv_robacc:.3f}')
assert adv_robacc > plain_robacc, '对抗训练应提升 PGD 下鲁棒精度'
print('✅ 对抗训练把鲁棒精度拉回来（注意干净精度可能略降 —— 鲁棒/准确率权衡）')

---
## ✏️ 练习区

每题先看题面，在 `TODO` 处补全（`raise NotImplementedError` 删掉），紧跟的自测 cell 应当通过。做完再看 📖 参考答案。

### ✏️ 练习 1：定向 FGSM

上面的 FGSM 是**非定向**的（只要分错）。实现**定向** FGSM：把样本推向**指定目标类** `target`。

提示：定向攻击要**降低**目标类的损失（朝目标类靠），所以沿 **负** 梯度方向走：$x_{adv}=x-\epsilon\cdot\mathrm{sign}(\nabla_x L(\cdot, target))$。

In [ ]:
def fgsm_targeted(model, X, target, eps):
    '''把 X 推向 target 类（target: 标量 0/1 或 (n,) 数组）。'''
    target = np.full(len(X), target) if np.isscalar(target) else np.asarray(target)
    # TODO: 算对「目标类标签」的损失梯度，沿负方向走 eps（让样本更像 target 类）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Xt = fgsm_targeted(clf, Xte, target=1, eps=0.8)
frac_to_1 = np.mean(clf.predict(Xt) == 1)
print(f'被推向类 1 的比例 = {frac_to_1:.3f}')
assert frac_to_1 > np.mean(clf.predict(Xte) == 1) + 0.1, '定向攻击应让更多样本落到目标类'
print('✅ 练习 1 通过')

### ✏️ 练习 2：手写 PGD 的一步（投影）

实现 PGD 的**单步更新** `pgd_step`：给当前 `Xa`、原点 `X`、预算 `eps`、步长 `alpha`，返回走一步并投影回 ε-球后的点。

In [ ]:
def pgd_step(model, Xa, X, y, eps, alpha):
    # TODO: (1) 算 grad_input；(2) 沿 sign 走 alpha；(3) clip 到 [X-eps, X+eps] 投影回 ε-球
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Xa0 = Xte.copy()
Xa1 = pgd_step(clf, Xa0, Xte, yte, eps=0.5, alpha=0.2)
assert np.all(np.abs(Xa1 - Xte) <= 0.5 + 1e-9), '投影后必须在 ε-球内'
def mean_loss(m, Xa):
    pp = np.clip(m.prob(Xa), 1e-9, 1-1e-9)
    return float(np.mean(-(yte*np.log(pp)+(1-yte)*np.log(1-pp))))
assert mean_loss(clf, Xa1) > mean_loss(clf, Xa0), '一步后损失应上升'
print('✅ 练习 2 通过：单步在 ε-球内且损失上升')

### ✏️ 练习 3：评测陷阱 —— 弱攻击高估鲁棒

用 `pgd` 的 `steps` 与 `restarts` 参数说明「弱攻击制造虚假安全感」：实现 `eval_robust(steps, restarts)` 返回该强度下的鲁棒精度，并验证 **steps=1, restarts=1 的弱攻击** 报出的鲁棒精度 **不低于** 强攻击（steps=40, restarts=5）。

In [ ]:
def eval_robust(steps, restarts, eps=0.5):
    # TODO: 用给定 steps/restarts 跑 pgd，返回 clf 在对抗样本上的精度
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
weak = eval_robust(steps=1, restarts=1)
strong = eval_robust(steps=40, restarts=5)
print(f'弱攻击报告鲁棒精度={weak:.3f}  |  强攻击={strong:.3f}')
assert weak >= strong - 1e-9, '弱攻击应高估鲁棒（精度更高或相等）'
print('✅ 练习 3 通过：弱攻击高估鲁棒 —— 评测必须用强攻击/自适应攻击')

### ✏️ 练习 4：对抗训练 vs 普通训练的鲁棒性对比

实现 `compare(eps)`：返回 `(普通模型PGD鲁棒精度, 对抗训练模型PGD鲁棒精度)`，验证后者更高。（可复用上面的 `adversarial_train` 与 `pgd`。）

In [ ]:
def compare(eps=0.5):
    # TODO: 训练普通模型与对抗训练模型，各自在 eps 下用 pgd 评鲁棒精度，返回二元组
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
plain_r, adv_r = compare(eps=0.5)
print(f'普通模型鲁棒={plain_r:.3f}  对抗训练鲁棒={adv_r:.3f}')
assert adv_r > plain_r, '对抗训练应更鲁棒'
print('✅ 练习 4 通过：对抗训练提升鲁棒性')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案：定向 FGSM —— 沿负梯度把样本推向目标类
def fgsm_targeted(model, X, target, eps):
    target = np.full(len(X), target) if np.isscalar(target) else np.asarray(target)
    g = model.grad_input(X, target.astype(float))    # 对目标标签的损失梯度
    return X - eps * np.sign(g)                       # 负方向：降低目标类损失

In [ ]:
# 练习 2 参考答案：PGD 单步 + 投影
def pgd_step(model, Xa, X, y, eps, alpha):
    g = model.grad_input(Xa, y)
    Xa = Xa + alpha * np.sign(g)
    return np.clip(Xa, X - eps, X + eps)

In [ ]:
# 练习 3 参考答案：用攻击强度参数评鲁棒
def eval_robust(steps, restarts, eps=0.5):
    Xa = pgd(clf, Xte, yte, eps=eps, alpha=eps/4, steps=steps, restarts=restarts)
    return acc(clf, Xa, yte)

In [ ]:
# 练习 4 参考答案：对抗训练 vs 普通
def compare(eps=0.5):
    plain = MLP().fit(Xtr, ytr)
    advm = adversarial_train(Xtr, ytr, eps=eps)
    pr = acc(plain, pgd(plain, Xte, yte, eps=eps), yte)
    ar = acc(advm,  pgd(advm,  Xte, yte, eps=eps), yte)
    return pr, ar

---
## 🧪 真实数据胶囊：梯度遮蔽制造的「虚假鲁棒」

复现 Athalye 2018 的核心教训：一个**随机化输入**的玩具「防御」能让朴素（确定性）攻击失效，看起来很鲁棒——但它并没有真的鲁棒，换个**自适应**思路（对随机性求期望梯度，EOT 的雏形）立刻戳破。

**这正是评测方法论的铁律**：攻击失败 ≠ 防御有效；必须用自适应攻击。

### 胶囊练习：实现「期望梯度」自适应攻击

玩具防御 `noisy_grad` 在求梯度前给输入加随机噪声。朴素 FGSM 用单次带噪梯度，会被噪声搅乱。
你要实现 `fgsm_eot`：对**多次**带噪梯度求**平均**（期望），再做 FGSM，从而绕过随机化。

In [ ]:
# 玩具随机化防御（已给好，非 TODO）：求梯度前给输入加高斯噪声（梯度遮蔽的一种）
def noisy_grad(model, X, y, sigma, seed):
    r = np.random.default_rng(seed)
    return model.grad_input(X + sigma * r.standard_normal(X.shape), y)

In [ ]:
def fgsm_eot(model, X, y, eps, sigma=0.3, n_samples=30):
    '''EOT 雏形：对 n_samples 次带噪梯度求平均，再取符号走一步。'''
    # TODO: 平均多次 noisy_grad（seed 用 0..n_samples-1），sign 后走 eps
    raise NotImplementedError

In [ ]:
# —— 胶囊自测 ——（先做上面的 TODO）
sigma = 0.3
# 朴素攻击：用单次带噪梯度（被随机化搅乱，攻击偏弱）
Xnaive = Xte + 0.5 * np.sign(noisy_grad(clf, Xte, yte, sigma, seed=123))
naive_succ = 1 - acc(clf, Xnaive, yte)
# 自适应攻击：EOT 平均梯度
Xeot = fgsm_eot(clf, Xte, yte, eps=0.5, sigma=sigma, n_samples=30)
eot_succ = 1 - acc(clf, Xeot, yte)
print(f'朴素攻击成功率={naive_succ:.3f}  |  自适应(EOT)成功率={eot_succ:.3f}')
assert eot_succ >= naive_succ, '自适应攻击应不弱于朴素攻击（戳破虚假鲁棒）'
print('✅ 胶囊通过：随机化“防御”挡得住朴素攻击，却挡不住自适应攻击 —— 虚假鲁棒')

In [ ]:
# 📖 胶囊参考答案
def fgsm_eot(model, X, y, eps, sigma=0.3, n_samples=30):
    g = np.zeros_like(X)
    for s in range(n_samples):
        g += noisy_grad(model, X, y, sigma, seed=s)
    g /= n_samples
    return X + eps * np.sign(g)

### 小结
- 对抗样本 = 在 ε 预算内让模型分错的微小扰动；根源是决策边界的局部脆弱 + 高维线性累积。
- **FGSM** 单步符号梯度（快、弱）；**PGD** 多步+投影+重启（强、是评测事实标准）。
- **对抗训练**（min-max：内层 PGD 生成、外层最小化）是最可靠的经验防御，代价是干净精度略降。
- 鲁棒性数字**离不开 ε**；**弱攻击/梯度遮蔽**会制造虚假安全感，**必须用强攻击 + 自适应攻击**评测。

下一站：**模块 02 · 数据投毒与后门** —— 攻击从推理期前移到训练期。